In [83]:
import archs4py

file = "/home/space/omics/datasets/archs4/human_gene_v2.latest.h5"

archs4py.ls(file)

data                      
│ expression            uint32 | (67186, 993191)
meta                      
│ genes                     
│   biotype               str    | (67186,)
│   ensembl_gene          str    | (67186,)
│   symbol                str    | (67186,)
│ info                      
│   author                str    | ()
│   contact               str    | ()
│   creation-date         str    | ()
│   laboratory            str    | ()
│   version               str    | ()
│ samples                   
│   alignedreads          float64 | (993191,)
│   channel_count         str    | (993191,)
│   characteristics_ch1   str    | (993191,)
│   contact_address       str    | (993191,)
│   contact_city          str    | (993191,)
│   contact_country       str    | (993191,)
│   contact_institute     str    | (993191,)
│   contact_name          str    | (993191,)
│   contact_zip           str    | (993191,)
│   data_processing       str    | (993191,)
│   extract_protocol_ch1  str    | (9

In [84]:
sample_fields = [
   'alignedreads',
   'channel_count',
   'characteristics_ch1',
   'contact_address',
   'contact_city',
   'contact_country',
   'contact_institute',
   'contact_name',
   'contact_zip',
   'data_processing',
   'extract_protocol_ch1',
   'geo_accession',
   'instrument_model',
   'last_update_date',
   'library_selection',
   'library_source',
   'library_strategy',
   'molecule_ch1',
   'organism_ch1',
   'platform_id',
   'relation',
   'sample',
   'series_id',
   'singlecellprobability',
   'source_name_ch1',
   'status',
   'submission_date',
   'taxid_ch1',
   'title',
   'type'
]

num_samples = 993191

In [85]:
from io import StringIO
from pathlib import Path

import pandas as pd
import requests


class GeneLengthFetcher:
    """Fetch gene lengths from Ensembl BioMart and cache in parquet."""
    
    def __init__(self, cache_file: str = "feature_lengths_cache.parquet"):
        """Initialize the gene length fetcher with parquet cache."""
        self.base_url = "http://www.ensembl.org/biomart/martservice"
        self.cache_file = Path(cache_file)
        self.cache_df = self._load_cache()
    
    def _load_cache(self) -> pd.DataFrame:
        """Load existing cache or create empty DataFrame."""
        if self.cache_file.exists():
            #logger.info(f"Loading gene length cache from {self.cache_file}")
            return pd.read_parquet(self.cache_file)
        else:
            #logger.info("Creating new gene length cache")
            return pd.DataFrame(columns=['ensembl_gene_id', 'feature_length'])
    
    def _save_cache(self):
        """Save cache to parquet file."""
        self.cache_df.to_parquet(self.cache_file, index=False)
        #logger.info(f"Saved gene length cache to {self.cache_file}")
    
    def fetch_feature_lengths(self, ensembl_ids: list[str]) -> pd.DataFrame:
        """
        Fetch gene lengths from Ensembl BioMart with caching.
        
        Args:
            ensembl_ids: List of Ensembl gene IDs
            
        Returns:
            DataFrame indexed by Ensembl ID with feature_length column
        """
        if not ensembl_ids:
            return pd.DataFrame(columns=['feature_length'], dtype='int64', index=pd.Index([], name='ensembl_gene_id'))
        
        # Check cache for existing lengths - vectorized approach
        ensembl_series = pd.Series(ensembl_ids)
        
        if not self.cache_df.empty:
            # Vectorized lookup
            cached_mask = ensembl_series.isin(self.cache_df['ensembl_gene_id'])
            uncached_ids = ensembl_series[~cached_mask].tolist()
        else:
            uncached_ids = ensembl_ids
        
        # Fetch uncached gene lengths
        if uncached_ids:
            #logger.info(f"Fetching gene lengths for {len(uncached_ids)} genes from Ensembl...")
            new_lengths = self._fetch_from_ensembl(uncached_ids)
            
            # Add to cache - vectorized concat
            if new_lengths:
                new_df = pd.DataFrame(list(new_lengths.items()), columns=['ensembl_gene_id', 'feature_length'])
                self.cache_df = pd.concat([self.cache_df, new_df], ignore_index=True)
                self._save_cache()
        
        # Create indexed cache for lookup
        cache_indexed = self.cache_df.set_index('ensembl_gene_id')
        
        # Query cache for all requested gene IDs, preserving order
        result_df = cache_indexed.reindex(ensembl_ids)
        
        # Don't raise error - just return the results with NaNs for missing genes
        # (genes that Ensembl couldn't find will have NaN values)
        result_df['feature_length'] = result_df['feature_length'].astype('Int64')  # Nullable integer
        return result_df
    
    def _fetch_from_ensembl(self, ensembl_ids: list[str]) -> dict[str, int]:
        """Fetch a chunk of gene lengths."""
        id_filter = ','.join(ensembl_ids)
        
        query = f"""<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE Query>
<Query virtualSchemaName="default" formatter="TSV" header="1" uniqueRows="0" count="" datasetConfigVersion="0.6">
    <Dataset name="hsapiens_gene_ensembl" interface="default">
        <Filter name="ensembl_gene_id" value="{id_filter}"/>
        <Attribute name="ensembl_gene_id"/>
        <Attribute name="start_position"/>
        <Attribute name="end_position"/>
    </Dataset>
</Query>"""
        
        response = requests.post(
            self.base_url,
            data={'query': query},
            timeout=60
        )
        response.raise_for_status()
        
        results = {}
        if response.text.strip():
            df = pd.read_csv(StringIO(response.text), sep='\t')
            df['feature_length'] = (df['Gene end (bp)'] - df['Gene start (bp)'] + 1).clip(lower=1)
            
            # Vectorized conversion to dict
            results = df.set_index('Gene stable ID')['feature_length'].astype(int).to_dict()
        
        return results

In [86]:
ensembl_genes = archs4py.meta.field(file, "ensembl_gene")
len(ensembl_genes)

67186

In [87]:
ensembl_genes

['ENSG00000000003',
 'ENSG00000000005',
 'ENSG00000000419',
 'ENSG00000000457',
 'ENSG00000000460',
 'ENSG00000000938',
 'ENSG00000000971',
 'ENSG00000001036',
 'ENSG00000001084',
 'ENSG00000001167',
 'ENSG00000001460',
 'ENSG00000001461',
 'ENSG00000001497',
 'ENSG00000001561',
 'ENSG00000001617',
 'ENSG00000001626',
 'ENSG00000001629',
 'ENSG00000001630',
 'ENSG00000001631',
 'ENSG00000002016',
 'ENSG00000002079',
 'ENSG00000002330',
 'ENSG00000002549',
 'ENSG00000002586',
 'ENSG00000002587',
 'ENSG00000002726',
 'ENSG00000002745',
 'ENSG00000002746',
 'ENSG00000002822',
 'ENSG00000002834',
 'ENSG00000002919',
 'ENSG00000002933',
 'ENSG00000003056',
 'ENSG00000003096',
 'ENSG00000003137',
 'ENSG00000003147',
 'ENSG00000003249',
 'ENSG00000003393',
 'ENSG00000003400',
 'ENSG00000003402',
 'ENSG00000003436',
 'ENSG00000003509',
 'ENSG00000003756',
 'ENSG00000003987',
 'ENSG00000003989',
 'ENSG00000004059',
 'ENSG00000004139',
 'ENSG00000004142',
 'ENSG00000004399',
 'ENSG00000004455',


In [88]:
feature_length_fetcher = GeneLengthFetcher()
feature_length_df = feature_length_fetcher.fetch_feature_lengths(ensembl_genes)

In [89]:
feature_length_df["feature_length"].isna().sum()

np.int64(1319)

In [91]:
import anndata as ad

chunk_size = 1000

for chunk_start, chunk_end in [(i, min(i + chunk_size, num_samples)) for i in range(0, num_samples, chunk_size)]:
    indices = list(range(chunk_start, chunk_end))

    expression_df = archs4py.data.index(file, indices)

    expression_df.index = ensembl_genes
    expression_df = expression_df.T
    sample_ids = expression_df.index.tolist()

    sample_meta_df = archs4py.meta.samples(file, sample_ids, meta_fields=sample_fields)

    adata = ad.AnnData(X=expression_df, var=feature_length_df, obs=sample_meta_df)
    adata = adata[adata.obs["singlecellprobability"] < 0.5]
    adata = adata[:, ~adata.var["feature_length"].isna()]
    
    print(adata)

100%|██████████| 1000/1000 [00:11<00:00, 86.46it/s]


View of AnnData object with n_obs × n_vars = 950 × 65867
    obs: 'alignedreads', 'channel_count', 'characteristics_ch1', 'contact_address', 'contact_city', 'contact_country', 'contact_institute', 'contact_name', 'contact_zip', 'data_processing', 'extract_protocol_ch1', 'geo_accession', 'instrument_model', 'last_update_date', 'library_selection', 'library_source', 'library_strategy', 'molecule_ch1', 'organism_ch1', 'platform_id', 'relation', 'sample', 'series_id', 'singlecellprobability', 'source_name_ch1', 'status', 'submission_date', 'taxid_ch1', 'title', 'type'
    var: 'feature_length'


100%|██████████| 1000/1000 [00:03<00:00, 321.61it/s]


View of AnnData object with n_obs × n_vars = 1000 × 65867
    obs: 'alignedreads', 'channel_count', 'characteristics_ch1', 'contact_address', 'contact_city', 'contact_country', 'contact_institute', 'contact_name', 'contact_zip', 'data_processing', 'extract_protocol_ch1', 'geo_accession', 'instrument_model', 'last_update_date', 'library_selection', 'library_source', 'library_strategy', 'molecule_ch1', 'organism_ch1', 'platform_id', 'relation', 'sample', 'series_id', 'singlecellprobability', 'source_name_ch1', 'status', 'submission_date', 'taxid_ch1', 'title', 'type'
    var: 'feature_length'


100%|██████████| 1000/1000 [00:03<00:00, 323.04it/s]


...

View of AnnData object with n_obs × n_vars = 362 × 65867
    obs: 'alignedreads', 'channel_count', 'characteristics_ch1', 'contact_address', 'contact_city', 'contact_country', 'contact_institute', 'contact_name', 'contact_zip', 'data_processing', 'extract_protocol_ch1', 'geo_accession', 'instrument_model', 'last_update_date', 'library_selection', 'library_source', 'library_strategy', 'molecule_ch1', 'organism_ch1', 'platform_id', 'relation', 'sample', 'series_id', 'singlecellprobability', 'source_name_ch1', 'status', 'submission_date', 'taxid_ch1', 'title', 'type'
    var: 'feature_length'


100%|██████████| 1000/1000 [00:03<00:00, 289.48it/s]


View of AnnData object with n_obs × n_vars = 948 × 65867
    obs: 'alignedreads', 'channel_count', 'characteristics_ch1', 'contact_address', 'contact_city', 'contact_country', 'contact_institute', 'contact_name', 'contact_zip', 'data_processing', 'extract_protocol_ch1', 'geo_accession', 'instrument_model', 'last_update_date', 'library_selection', 'library_source', 'library_strategy', 'molecule_ch1', 'organism_ch1', 'platform_id', 'relation', 'sample', 'series_id', 'singlecellprobability', 'source_name_ch1', 'status', 'submission_date', 'taxid_ch1', 'title', 'type'
    var: 'feature_length'


 20%|██        | 202/1000 [00:00<00:03, 257.19it/s]


KeyboardInterrupt: 